# 01 - 模型架构逐层解析

本 notebook 深入讲解 MiniMind-O 的模型结构，目标：
1. 理解 `model_minimind.py` 中的 Thinker（文本主干）
2. 理解 `model_omni.py` 中的 Talker、投影层、多模态编码器注入
3. 识别关键设计选择与潜在问题

阅读前建议先打开源码：
- `model/model_minimind.py`
- `model/model_omni.py`

## 1. Thinker：MiniMind 文本主干

`MiniMindModel` 是一个标准的 decoder-only Transformer，包含：
- `embed_tokens`：词表嵌入
- `layers`：N 个 `MiniMindBlock`
- `norm`：RMSNorm
- `freqs_cos / freqs_sin`：预计算的 RoPE 位置编码

关键配置（默认 113M）：
- `hidden_size=768`
- `num_hidden_layers=8`
- `num_attention_heads=8`
- `num_key_value_heads=4`（GQA，减少 KV 缓存）
- `head_dim=96`
- `intermediate_size=ceil(768*pi/64)*64=3840`

In [ ]:
import os, sys
ROOT = "/home/genesis/Projects/minimind-o"
os.chdir(ROOT)
sys.path.insert(0, ROOT)

import torch
from model.model_minimind import MiniMindConfig, MiniMindModel

cfg = MiniMindConfig(hidden_size=768, num_hidden_layers=8, num_attention_heads=8, num_key_value_heads=4)
thinker = MiniMindModel(cfg)
print(thinker)
print(f"\n参数数量: {sum(p.numel() for p in thinker.parameters())/1e6:.2f} M")

### 1.1 RMSNorm

代码见 `model_minimind.py:52-63`。

公式：
```
RMSNorm(x) = x / sqrt(mean(x^2) + eps) * weight
```

注意：`forward` 中先 cast 到 float32 做 norm，再 cast 回原始 dtype。这是数值稳定性常规做法。

### 1.2 RoPE 位置编码

代码见 `model_minimind.py:64-86`。

默认 `rope_theta=1e6`，支持外推。如果启用 `inference_rope_scaling`，则使用 YaRN 扩展长上下文。

**关键注意点**：
- `precompute_freqs_cis` 在模型初始化时计算全长的 cos/sin 并注册为 buffer。
- `MiniMindModel.forward` 中有一行防御代码：`if self.freqs_cos[0, 0] == 0`，用于在 `transformers>=5.x` meta-device 初始化导致 buffer 被清空时重新计算。
- 这一判断依赖 `freqs_cos[0,0]` 的数值，若 RoPE base 恰好让第一个值为 0，会触发不必要的重算。

In [ ]:
# 观察 RoPE buffer 形状
print(f"freqs_cos shape: {thinker.freqs_cos.shape}")  # (max_position_embeddings, head_dim)
print(f"first 5 values: {thinker.freqs_cos[0, :5]}")
print(f"freqs_cos[0,0] == 0: {thinker.freqs_cos[0, 0].item() == 0}")

### 1.3 Attention

代码见 `model_minimind.py:93-136`。

关键点：
- Q/K/V 投影后都会做 `RMSNorm`（q_norm / k_norm），这是 Llama3/Qwen 等模型的常见做法。
- 使用 `repeat_kv` 将 K/V 从 `num_key_value_heads` 复制到 `num_attention_heads`。
- FlashAttention 条件判断较复杂：`self.flash and seq_len > 1 and (not causal or no past_kv) and (mask all ones)`。

**潜在问题**：
- 当 `past_key_value` 存在时（use_cache=True），会走手动 attention 路径而不是 FlashAttention。单次 decode step 序列长度为 1，开销不大。
- 如果 `attention_mask` 中存在 0（padding），会触发手动路径； FlashAttention 的 `is_causal=True` 与自定义 mask 不兼容。

In [ ]:
# 跑一个简单的前向，观察输出结构
x = torch.randint(0, cfg.vocab_size, (2, 16))  # batch=2, seq=16
with torch.no_grad():
    hidden, past_kvs, aux = thinker(x)
print(f"hidden shape: {hidden.shape}")
print(f"past_kvs len: {len(past_kvs)}")
print(f"aux loss: {aux}")

### 1.4 FFN 与 MoE

代码见 `model_minimind.py:138-178`。

- `FeedForward` 是标准的 SwiGLU：`down_proj(silu(gate_proj(x)) * up_proj(x))`。
- `MOEFeedForward` 是本地 FFN MoE（非 Expert Parallel）。

**注意**：当前所有已完成的 full train 都是 `use_moe=0`。MoE 会增加路由不稳定性和显存压力，不建议在 audio 问题未收敛前启用。

## 2. Omni：多模态外壳

`MiniMindOmni` 继承自 `MiniMindForCausalLM`，新增：
- `talker`：TalkerModule
- `audio_proj` / `vision_proj`：投影层
- `audio_encoder` / `vision_encoder`：冻结的预训练编码器

`thinker` 被设置为 `self.model` 的别名：`self.thinker = self.model`。

In [ ]:
from model.model_omni import OmniConfig, MiniMindOmni

omni_cfg = OmniConfig(
    hidden_size=768,
    num_hidden_layers=8,
    num_attention_heads=8,
    num_key_value_heads=4,
    use_moe=False,
)

# 实例化；编码器路径不存在会打印 warning，不影响结构分析
omni = MiniMindOmni(omni_cfg)
print(f"thinker is model: {omni.thinker is omni.model}")
print(f"talker layers: {len(omni.talker.layers)}")
print(f"audio_proj params: {sum(p.numel() for p in omni.audio_proj.parameters())/1e6:.3f} M")
print(f"vision_proj params: {sum(p.numel() for p in omni.vision_proj.parameters())/1e6:.3f} M")

### 2.1 TalkerModule 结构

代码见 `model_omni.py:93-119`。

Talker 是一个独立的 MiniMindBlock 堆叠：
- `layers`：4 层 Transformer block（默认）
- `embed_tokens`：`TalkerEmbedding`，对 8 层 RVQ code 分别做 adapter 再平均
- `lm_head`：`TalkerHead`，对每层 RVQ 输出一个 adapter 修正的 logit
- `codec_proj`：把 audio embedding 映射到 talker hidden
- `embed_proj`：把 Thinker hidden 映射到 talker hidden
- `text_scale / audio_scale`：可学习缩放参数（默认 3.0 / 1.0）
- `spk_proj`：说话人嵌入投影

Talker 的输入是：
```
hidden = embed_proj(bridge_states) * text_scale + codec_proj(audio_embed) * audio_scale
```
其中 `bridge_states` 来自 Thinker 的 `bridge_layer` 输出。

### 2.2 TalkerEmbedding / TalkerHead

这两个组件都使用了 `num_layers=8` 个 adapter：

- `TalkerEmbedding`：`base + adapter_i(code_i)` 后平均。代码见 `model_omni.py:73-81`。
- `TalkerHead`：`base_out + adapter_i(x)` 得到 8 个 logits。代码见 `model_omni.py:62-70`。

**设计解读**：
- 对 8 层 RVQ 分别做轻量 adapter，让每层码本有独立的预测头，同时共享主体。

**潜在问题**：
- `TalkerEmbedding` 在 forward 中对 `i` 做 `range(len(self.adapters))`，但 `base_out` 维度是 `(B, 8, D)`，`self.adapters[i](x[:, i, :])` 正确索引第 i 层 code。
- `TalkerHead` 返回 list，后续处理需要特别注意 list 的顺序与 RVQ 层索引对应。

In [ ]:
# 打印 Talker 各子模块参数
talker = omni.talker
submodules = {
    "layers": sum(p.numel() for p in talker.layers.parameters()),
    "lm_head": sum(p.numel() for p in talker.lm_head.parameters()),
    "embed_tokens": sum(p.numel() for p in talker.embed_tokens.parameters()),
    "codec_proj": sum(p.numel() for p in talker.codec_proj.parameters()),
    "embed_proj": sum(p.numel() for p in talker.embed_proj.parameters()),
}
for k, v in submodules.items():
    print(f"{k:15s}: {v/1e6:.3f} M")
print(f"total talker : {sum(submodules.values())/1e6:.2f} M")

### 2.3 音频编码器注入

流程：
1. `encode_audio_inputs(audio_inputs, audio_lens)`：把 fbank 喂给冻结的 SenseVoice，得到帧级特征 `(B, T, 512)`。
2. `audio_proj` 把 512 维映射到 Thinker hidden（默认 768）。
3. `inject_audio_features(tokens, h, audio_feats, seqlen)`：在 `audio_token_id` 的位置把音频特征替换进去。

代码见 `model_omni.py:154-198`。

**关键注意点**：
- `encode_audio_inputs` 用 `batch_mask = audio_inputs.flatten(1).any(1)` 跳过全 0 的 dummy 样本，但 `audio_inputs` 在 dataset 中无音频时会被填成 `torch.zeros(1, 1, 560)`，`any(1)` 为 False，返回 None。
- `inject_audio_features` 是 Python 循环逐样本处理，batch 较大时效率不高。
- 音频特征注入只发生在 `start_pos == 0`（训练阶段），推理 streaming 时由 prompt 中的 `<|audio_pad|>` token 占位。

### 2.4 视觉编码器注入

流程：
1. `encode_image_inputs(pixel_values)`：把图片喂给冻结的 SigLIP2，得到 patch 特征。
2. `vision_proj` 映射到 Thinker hidden。
3. `count_vision_proj(tokens, h, vision_tensors, seqlen)`：在 `<|image_pad|>` 占位符位置插入图像特征。

代码见 `model_omni.py:215-260`。

**关键注意点**：
- `vision_tensors` 形状可能是 `(B, N_img, image_token_len, hidden)`，需要 `unsqueeze(1)` 后按 `num` 维度循环处理。
- 多图场景下，`k` 计数器按样本内图片数量递增；当前代码默认 `image_token_len=64`。
- `count_vision_proj` 会把结果截断到 `seqlen`，防止超长序列。

## 3. MiniMindOmni.forward 总览

代码见 `model_omni.py:262-373`。

输入 `input_ids` 的形状有两种：
- `(B, T)`：训练时，text_ids 等于 input_ids，audio_ids 自动填充 pad_token。
- `(B, 9, T)`：推理 streaming 时，前 8 层是 audio codes，第 9 层是 text ids。

训练阶段流程：
1. Thinker 前向，注入 audio/vision 特征。
2. 在 `bridge_layer` 处保存 `bridge_states`。
3. Talker 前向：`embed_proj(bridge_states) * text_scale + codec_proj(audio_embed) * audio_scale`。
4. 输出 `text_logits` 和 8 个 `audio_logits`。

**注意**：
- `aux_loss` 包含 dummy gradient（`sum(p.sum() * 0`），目的是让 optimizer 看到 projectors / spk_proj / adapters 的梯度，即使 MoE aux=0。
- `logits_to_keep` 支持只计算最后 N 个 token 的 logits，用于推理加速。

In [ ]:
# 一个训练形状的前向 smoke
omni = omni.eval().cuda() if torch.cuda.is_available() else omni.eval()
dummy_ids = torch.randint(0, omni_cfg.vocab_size, (2, 32), device=next(omni.parameters()).device)
with torch.no_grad():
    out = omni(dummy_ids)
print(f"text_logits shape: {out.logits.shape}")
print(f"audio_logits list len: {len(out.audio_logits)}")
print(f"audio_logits[0] shape: {out.audio_logits[0].shape}")

## 4. 推理生成：stream_generate

代码见 `model_omni.py:375-468`。

MiniMindOmni 重写了 `generate`，采用流式生成：
- 每次生成一个 text token，同时生成 8 个 audio code。
- audio 比 text 延迟：第 i 层 audio code 在第 `i+1` 个 text token 之后才开始生成。
- audio stop token（`>=2048`）用于判断某层 RVQ 是否结束。

**关键设计**：
- `think_end_ids`：默认 `[26, 234, 234]`，对应 tokenizer 中的 `</think>` 序列。开启 thinking 模式后，audio 只在 think 结束后生成。
- `audio_buffer`：保存历史 audio codes，作为下一步的 audio 输入。
- `ref_codes` / `spk_emb`：支持音色克隆。

**已知问题与调试要点**：
1. **采样数值稳定性**：`logits.float()` 在修复前是缺失的，fp16 推理时会出现非有限概率。当前代码已修复（`model_omni.py:409` 和 `445`）。
2. **audio 延迟理解**：新手容易困惑为什么第一个 text token 没有 audio frame。注意 `audio_step = step - 1`，第一层 audio 从第二个 text step 开始。
3. **特殊 token 过滤**：推理时会将 `>=2049` 的 code 替换为 0 再喂给 Mimi（`eval_omni.py:81`），因为 Mimi codebook 只有 2048 个有效码。

## 5. 本模块识别的关键问题清单

| 问题 | 位置 | 影响 | 建议 |
| --- | --- | --- | --- |
| RoPE 重算判断依赖 `freqs_cos[0,0]==0` | `model_minimind.py:218` | 极小概率误触发 | 可改为检查 buffer 是否全 0 |
| `inject_audio_features` 用 Python 循环 | `model_omni.py:178-198` | batch 大时 CPU/GPU 同步开销 | 可尝试向量化 |
| `count_vision_proj` 也是 Python 循环 | `model_omni.py:240-260` | 同理 | 可尝试向量化 |
| TalkerHead 返回 list，容易索引混乱 | `model_omni.py:62-70` | 训练/推理时需严格对齐 RVQ 层 | 建议封装成 tensor 或 namedtuple |
| `aux_loss` 含大量 dummy gradient | `model_omni.py:366-367` | 代码可读性差 | 可用 register_full_backward_hook 替代 |
| FlashAttention 路径判断复杂 | `model_minimind.py:127` | 自定义 mask / use_cache 时降级手动 attention | 注意 profile 验证 |
| audio_token / image_token 注入只发生在 start_pos==0 | `model_omni.py:286,289` | streaming 多轮对话时特征注入位置需靠 prompt 控制 | 理解 prompt 构造逻辑 |

## 6. 练习

1. 修改 `bridge_layer`（例如从 3 改为 5），观察 Talker 输入来自 Thinker 哪一层。
2. 打印 `MiniMindOmni.forward` 中 `bridge_states` 与 `h_thinker` 的 L2 norm，比较两者差异。
3. 在 `stream_generate` 中打印每个 step 的 `audio_step` 和 `audio_codes[i]`，验证延迟 1 step 的规则。